# Tutorial 6 — Value function iteration

**Source:** `../PS3/Problem_Set_3_Q3.py`

**In class:** PS3 Question 3(d)–(f); use part (g) if time remains.

Run cells from top to bottom. At each pause, predict the result before running the next cell. Plots appear below their cells; the source scripts remain the full resolution reference.


## 1. The Bellman problem

$$V(k)=\max_{k'}\left\{\log\bigl(Ak^\alpha+(1-\delta)k-k'\bigr)+\beta V(k')\right\}.$$

The current state is $k$; the choice is next period's $k'$. Each row of a payoff matrix will represent one current $k$, and each column one candidate $k'$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
A, BETA, DELTA, ALPHA = 1.0, .90, .10, .30
N_GRID, TOL = 150, 1e-6  # live grid; source script uses 500 nodes
print('Discount factor:', BETA)


In [ ]:
def k_bar(alpha, A=A, beta=BETA, delta=DELTA):
    """
    Steady-state capital, part (c).

    The Euler equation at a steady state reads 1 = beta[alpha A k^(alpha-1) + 1 - delta],
    so alpha A k^(alpha-1) = 1/beta - (1 - delta), which inverts directly.
    """
    return (alpha * A / (1 / beta - (1 - delta))) ** (1 / (1 - alpha))


In [ ]:
kb = k_bar(ALPHA)
print('Analytical steady-state capital:', round(kb, 4))


## 2. Try a three-point grid first

Feasible consumption is $c(k,k')=Ak^\alpha+(1-\delta)k-k'$. Negative or zero consumption cannot be logged. Mark such choices as $-\infty$ so `argmax` never selects them.


In [ ]:
tiny = np.array([.25, .75, 1.25])
resources = A*tiny[:, None]**ALPHA + (1-DELTA)*tiny[:, None] - tiny[None, :]
payoff = np.where(resources > 0, np.log(np.maximum(resources, 1e-12)), -np.inf)
print('Consumption matrix:\n', np.round(resources, 3))
print('Payoff matrix:\n', np.round(payoff, 3))


## 3. One Bellman update

Starting from $V_0=0$, add $\beta V_0(k')$ to every column of the payoff matrix. Take the best column in each row. The selected column is the savings policy.


In [ ]:
V0 = np.zeros(len(tiny))
M = payoff + BETA*V0[None, :]
choice = M.argmax(axis=1)
V1 = M[np.arange(len(tiny)), choice]
print('Chosen next capital:', tiny[choice])
print('New value:', np.round(V1, 3))


## 4. Iterate until the value function settles

The full function repeats that one update. The flow payoff matrix is built once. The stopping rule checks $\|V_{j+1}-V_j\|_\infty<10^{-6}$; the value error is bounded by roughly this change divided by $1-\beta$.


In [ ]:
def solve(alpha, n=N_GRID, tol=TOL, A=A, beta=BETA, delta=DELTA):
    """
    Value function iteration on a grid.

    The whole method is three lines of arithmetic. Everything else is bookkeeping.
    Build the matrix of period payoffs once, outside the loop -- it does not depend
    on the value function, so recomputing it every iteration is pure waste.
    """
    kb = k_bar(alpha, A, beta, delta)
    grid = np.linspace(1e-3, 2 * kb, n)

    # resources[i, j] = what is left to consume at k_i after choosing k' = k_j
    resources = A * grid[:, None] ** alpha + (1 - delta) * grid[:, None] - grid[None, :]
    feasible = resources > 0
    # -inf on infeasible choices, so the maximiser never selects them
    payoff = np.where(feasible, np.log(np.where(feasible, resources, 1.0)), -np.inf)

    V = np.zeros(n)
    for iteration in range(20_000):
        M = payoff + beta * V[None, :]        # value of every (k, k') pair
        choice = M.argmax(axis=1)             # best k' for each k
        V_new = M[np.arange(n), choice]
        gap = np.max(np.abs(V_new - V))
        V = V_new
        if gap < tol:
            break

    policy = grid[choice]
    consumption = A * grid**alpha + (1 - delta) * grid - policy
    return grid, V, policy, consumption, iteration + 1, gap, kb


In [ ]:
grid, V, policy, consumption, iters, gap, kb = solve(ALPHA)
print(f'{iters} iterations; final change {gap:.2e}; value error bound ≈ {gap/(1-BETA):.2e}')


## 5. Read the policy

Where $k'(k)>k$, capital rises. A crossing of the $45^\circ$ line is the numerical steady state. Its distance from the analytical $\bar k$ is limited by the grid spacing.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(grid, policy, label=r"$k'(k)$")
ax.plot(grid, grid, '--', color='0.4', label='45° line')
ax.axvline(kb, color='C3', ls=':')
ax.set(xlabel='$k$', ylabel="$k'$", title='Policy and fixed point')
ax.legend(frameon=False); plt.show()
print('Grid spacing near steady state:', grid[1]-grid[0])


## 6. Generate a path

The policy is defined on grid nodes. Interpolation lets us apply it to a current $k$ between nodes. Starting at $0.5\bar k$, repeatedly choose the next $k$.


In [ ]:
def simulate(grid, policy, k0, T=50):
    """Iterate the policy forward, interpolating between grid points."""
    path = np.empty(T + 1)
    path[0] = k0
    for t in range(T):
        path[t + 1] = np.interp(path[t], grid, policy)
    return path


In [ ]:
k_path = simulate(grid, policy, .5*kb, T=50)
c_path = A*k_path[:-1]**ALPHA+(1-DELTA)*k_path[:-1]-k_path[1:]
print('k0, k1, k50:', np.round(k_path[[0, 1, 50]], 4))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(k_path); axes[0].axhline(kb, color='0.5', ls='--')
axes[1].plot(c_path, color='C1')
axes[0].set_title('Capital'); axes[1].set_title('Consumption')
for ax in axes: ax.set_xlabel('period')
plt.tight_layout(); plt.show()


## 7. What changes when $\alpha$ changes?

Run part (g) only if time permits. Diminishing returns are stronger for smaller $\alpha$, so convergence should be faster. Predict the ordering before running.


In [ ]:
for alpha in (.10, .30, .60):
    g, _, pol, _, _, _, steady = solve(alpha)
    path = simulate(g, pol, .5*steady)
    within = np.where(np.abs(path-steady)/steady < .10)[0]
    print(f'alpha={alpha:.2f}: first period within 10% = {within[0] if len(within) else "beyond 50"}')
